# Simple RAG Prototype with Ollama

This notebook implements a basic Retrieval-Augmented Generation (RAG) system for job postings.

It loads pre-built FAISS index and metadata, retrieves relevant documents for a user query,
and sends them as context to a local LLM running through Ollama.

**Prerequisites:**
- Pre-computed FAISS index and metadata in `../data/precomputed/`
- Ollama installed and running locally with a model pulled

## 0. Installing Ollama & Choosing a Model

### Installation

**Linux:**
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**macOS:**
Download from [ollama.com/download](https://ollama.com/download) or use Homebrew:
```bash
brew install ollama
```

**Windows:**
Download the installer from [ollama.com/download](https://ollama.com/download).

After installing, start the Ollama server:
```bash
ollama serve
```

### Which Model to Use (CPU, no GPU)

For CPU-only machines, smaller quantized models are the way to go.
The key constraint is RAM, not compute: the model needs to fit in memory.

| Model | Size on Disk | RAM Needed | Speed (CPU) | Quality | Best For |
|---|---|---|---|---|---|
| `gemma3:4b` | ~3 GB | ~5 GB | Fast | Good | Quick prototyping |
| `mistral` (7B) | ~4 GB | ~6 GB | Moderate | Very good | Best balance for CPU |
| `llama3.2:3b` | ~2 GB | ~4 GB | Fast | Decent | Low-RAM machines |
| `phi4-mini` (3.8B) | ~2.5 GB | ~5 GB | Fast | Good | Compact and capable |

**Recommendation for this prototype:** `mistral` gives the best quality-to-speed ratio on CPU.
If your machine has less than 8 GB RAM, go with `llama3.2:3b` instead.

Pull the model (one-time download):
```bash
ollama pull mistral
```

Verify it works:
```bash
ollama run mistral "Say hello in one sentence."
```

## 1. Install Dependencies

In [1]:
!pip install -q sentence-transformers faiss-cpu requests tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install tf-keras


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install einops


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

In [4]:
import os
import numpy as np
import pandas as pd
import faiss
import requests
import json
import textwrap

from sentence_transformers import SentenceTransformer

# ---------- CONFIG ----------
DATA_DIR = '../data/precomputed/rag'

METADATA_FILE = os.path.join(DATA_DIR, 'job_metadata.csv')
FAISS_INDEX   = os.path.join(DATA_DIR, 'faiss_index.bin')

ID_COL     = 'job_id'
DOC_COL    = 'document'

EMBED_MODEL = 'nomic-ai/nomic-embed-text-v1'

OLLAMA_URL   = 'http://localhost:11434'
OLLAMA_MODEL = 'mistral'

TOP_K          = 5
MAX_DOC_CHARS  = 1500
# ----------------------------

## 3. Load FAISS Index & Metadata

In [5]:
# load metadata
metadata = pd.read_csv(METADATA_FILE)
print(f'Metadata loaded: {len(metadata):,} documents')

# load FAISS index
index = faiss.read_index(FAISS_INDEX)
print(f'FAISS index loaded: {index.ntotal:,} vectors, dimension={index.d}')

# sanity check
assert len(metadata) == index.ntotal, 'Metadata and index row counts do not match!'
print('Sanity check passed.')

Metadata loaded: 20,000 documents
FAISS index loaded: 20,000 vectors, dimension=768
Sanity check passed.


## 4. Load Embedding Model

In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

embed_model = SentenceTransformer(
    EMBED_MODEL,
    trust_remote_code=True,
    device=device
)

print('Embedding model loaded.')

Device: cpu


pytorch_model.bin:   0%|          | 0.00/547M [00:00<?, ?B/s]

<All keys matched successfully>


Embedding model loaded.


## 5. Verify Ollama Connection

In [19]:
def check_ollama():
    """Verify that Ollama is running and the model is available."""
    try:
        resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
        resp.raise_for_status()
        models = [m['name'] for m in resp.json().get('models', [])]
        print(f'Ollama is running. Available models: {models}')

        # check if our target model is pulled
        match = any(OLLAMA_MODEL in m for m in models)
        if match:
            print(f'Model "{OLLAMA_MODEL}" is available.')
        else:
            print(f'WARNING: Model "{OLLAMA_MODEL}" not found.')
            print(f'Run: ollama pull {OLLAMA_MODEL}')
        return match

    except requests.ConnectionError:
        print('ERROR: Cannot connect to Ollama.')
        print('Make sure Ollama is running: ollama serve')
        return False

ollama_ready = check_ollama()

Ollama is running. Available models: ['mistral:latest']
Model "mistral" is available.


## 6. Core RAG Functions

In [8]:
def retrieve(query, top_k=TOP_K):
    """
    Encode the query and search FAISS for the top-k most similar documents.

    Returns a list of dicts with job_id, document text, and similarity score.
    """
    query_embedding = embed_model.encode(
        [f'search_query: {query}'],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    scores, indices = index.search(query_embedding, k=top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        row = metadata.iloc[idx]
        results.append({
            'rank': rank,
            'job_id': row[ID_COL],
            'score': float(score),
            'document': str(row[DOC_COL])[:MAX_DOC_CHARS]
        })

    return results

In [9]:
def build_prompt(query, retrieved_docs):
    """
    Build the prompt that combines retrieved context with the user's question.
    """
    context_block = '\n\n'.join(
        f'--- Job {doc["rank"]} (score: {doc["score"]:.3f}) ---\n{doc["document"]}'
        for doc in retrieved_docs
    )

    system_prompt = (
        'You are a helpful job market assistant. '
        'Answer the user\'s question using ONLY the job postings provided below. '
        'If the answer cannot be found in the provided postings, say so clearly. '
        'Be concise and specific. Reference job details when possible.'
    )

    user_prompt = (
        f'Here are the most relevant job postings:\n\n'
        f'{context_block}\n\n'
        f'---\n\n'
        f'Question: {query}'
    )

    return system_prompt, user_prompt

In [10]:
def generate(system_prompt, user_prompt):
    """
    Send the prompt to Ollama and return the generated response.
    Uses the /api/chat endpoint with streaming disabled for simplicity.
    """
    payload = {
        'model': OLLAMA_MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ],
        'stream': False,
        'options': {
            'temperature': 0.3,
            'num_predict': 512
        }
    }

    resp = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json=payload,
        timeout=120
    )
    resp.raise_for_status()

    data = resp.json()
    return data['message']['content']

In [11]:
def ask(query, top_k=TOP_K, verbose=True):
    """
    Full RAG pipeline: retrieve relevant docs, build prompt, generate answer.
    """
    # Step 1: Retrieve
    docs = retrieve(query, top_k=top_k)

    if verbose:
        print(f'Query: {query}')
        print(f'Retrieved {len(docs)} documents\n')
        for doc in docs:
            print(f'  Rank {doc["rank"]} | Score: {doc["score"]:.4f} | Job ID: {doc["job_id"]}')
        print()

    # Step 2: Build prompt
    system_prompt, user_prompt = build_prompt(query, docs)

    # Step 3: Generate
    if verbose:
        print('Generating answer (this may take a moment on CPU)...\n')

    answer = generate(system_prompt, user_prompt)

    if verbose:
        print('Answer:')
        print('-' * 60)
        print(answer)
        print('-' * 60)

    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': docs
    }

## 7. Test Retrieval Only

Before involving the LLM, verify that retrieval returns sensible results.

In [12]:
test_query = 'remote machine learning engineer with python'

results = retrieve(test_query)

for doc in results:
    print('=' * 80)
    print(f'Rank {doc["rank"]} | Score: {doc["score"]:.4f} | Job ID: {doc["job_id"]}')
    print(doc['document'][:500])
    print()

Rank 1 | Score: 0.6570 | Job ID: 3871631334
search_document: Job Title: Machine Learning Engineer
Company: NLB Services
Work Type: Full-time
Remote: No
Location: Dallas, TX
Domain: Technology
Description: Job Title: Python AI/ MLType: FulltimeLocation: Dallas, TX Python/AI-ML:Hands on experience with Python, Streamlit, Fastapi (minimum 2+ max 6 years)Hands on experience in developing neural networks using Tensorflow or Pytorch frameworkHands on experience with NLP (NLTK, Spacy, BERT, SBERT models)Hands on experience with vector database (

Rank 2 | Score: 0.6423 | Job ID: 3885105100
search_document: Job Title: MLE/AI Engineer Summer Intern
Experience Level: Internship
Work Type: Full-time
Remote: No
Location: San Francisco Bay Area
Domain: Education
Description: Company DescriptionWe are a stealth startup, an early-stage company based in the San Francisco Bay Area, operating in stealth mode to protect sensitive information and carefully manage our public image. Role DescriptionWe are s

## 8. Full RAG Query

Run the complete pipeline: retrieval + generation through Ollama.

On CPU, expect 10-30 seconds per answer depending on the model and response length.

In [13]:
result = ask('What remote data science jobs are available that require Python and SQL?')

Query: What remote data science jobs are available that require Python and SQL?
Retrieved 5 documents

  Rank 1 | Score: 0.6632 | Job ID: 3888826255
  Rank 2 | Score: 0.6530 | Job ID: 3887572653
  Rank 3 | Score: 0.6475 | Job ID: 3884434978
  Rank 4 | Score: 0.6473 | Job ID: 3884436863
  Rank 5 | Score: 0.6460 | Job ID: 3887885615

Generating answer (this may take a moment on CPU)...

Answer:
------------------------------------------------------------
 Based on the provided job postings, there are two remote data science jobs that require Python and SQL:

1. Job 2 (Data Scientist or Senior) at Progressive Insurance: This role requires proficiency in Python (or similar) development, debugging, and toolchain, as well as experience with SQL to process large volumes of structured and unstructured data.

2. Job 3 (Senior Data Analyst - AI Training, Remote, Contract) at Outlier: This contract position requires a minimum of 2 years of recent work experience in SQL, and the job involves optim

In [14]:
result = ask('Are there any entry-level software engineering positions?')

Query: Are there any entry-level software engineering positions?
Retrieved 5 documents

  Rank 1 | Score: 0.6514 | Job ID: 3885846111
  Rank 2 | Score: 0.6440 | Job ID: 3885806695
  Rank 3 | Score: 0.6342 | Job ID: 3888979085
  Rank 4 | Score: 0.6274 | Job ID: 3887102462
  Rank 5 | Score: 0.6269 | Job ID: 3887892888

Generating answer (this may take a moment on CPU)...

Answer:
------------------------------------------------------------
 Yes, there are entry-level software engineering positions available. Job 2 by Capital One and Job 5 by Pluralsight both list their experience level as "Entry level".
------------------------------------------------------------


In [15]:
result = ask('Which companies are hiring for healthcare-related roles?')

Query: Which companies are hiring for healthcare-related roles?
Retrieved 5 documents

  Rank 1 | Score: 0.6396 | Job ID: 3720384155
  Rank 2 | Score: 0.6329 | Job ID: 3884839022
  Rank 3 | Score: 0.6308 | Job ID: 3886214453
  Rank 4 | Score: 0.6249 | Job ID: 3884897030
  Rank 5 | Score: 0.6225 | Job ID: 3886456196

Generating answer (this may take a moment on CPU)...

Answer:
------------------------------------------------------------
 The companies hiring for healthcare-related roles are Concord Medical Group, Ciox Health, Pride Health, and B. Braun Medical Inc. (US). Specifically, Concord Medical Group is looking for a Healthcare Recruiter, Ciox Health is seeking Area Lead Health Information Specialists (both onsite and hybrid), Pride Health is hiring a Medical Technician, and B. Braun Medical Inc. (US) is looking for a Spec II, Healthcare Solutions.
------------------------------------------------------------
